# HV-013 — Final Engineering Assessment

**Technical gate:** FA-001

This notebook integrates the complete analyst-visible validation chain into a final requirement-by-requirement engineering assessment.

## 1. Final Engineering Question

> Is the sensor suitable for the intended HVAC measurement application, under the tested operating conditions, after calibration and with quantified measurement uncertainty?

The answer is separated into:

1. full public-spec compliance;
2. steady-state/static HVAC suitability;
3. dynamic HVAC suitability.

## 2. Reproducible Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "config" / "public" / "lab_spec_v1.json").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate project root.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.final_assessment import (
    build_application_risk_matrix,
    build_compliance_matrix,
    linearity_metric,
    load_final_evidence,
    load_frozen_c1_offset,
    load_public_spec,
    load_static_characterization,
    matched_static_hysteresis,
    summarize_characterization_tests,
    temperature_influence_metric,
)

tables_dir = project_root / "reports" / "tables"
figures_dir = project_root / "reports" / "figures"
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")

## 3. FA-01 — Public Acceptance-Limit Audit

All limits are loaded unchanged from the imported public laboratory specification.

In [ ]:
public_spec = load_public_spec(project_root)
limits = public_spec["dut_public_spec"]["acceptance_limits"]

display(pd.DataFrame(
    [{"Requirement": k, "Public_Value": v} for k, v in limits.items()]
))

expected_keys = {
    "static_accuracy_abs_rh_percent",
    "repeatability_sigma_max_rh_percent",
    "hysteresis_max_rh_percent",
    "linearity_max_rh_percent",
    "temperature_influence_max_rh_percent_per_c",
    "t90_max_s",
    "settling_band_abs_rh_percent",
    "drift_max_rh_percent_per_year",
}
assert set(limits) == expected_keys

print("FA-01 public acceptance-limit audit: PASS")

## 4. FA-02 — Static Characterization Metrics

C1 is frozen and is not refit.

The characterization dataset is used to compute:

- strict maximum within-Test_ID repeatability;
- matched static hysteresis;
- project-defined best-fit linearity;
- RH-conditioned temperature influence.

In [ ]:
c1_offset = load_frozen_c1_offset(project_root)
char_df, char_matrix_df = load_static_characterization(project_root)

static_test_summary_df = summarize_characterization_tests(
    char_df,
    char_matrix_df,
    c1_offset,
)

assert len(static_test_summary_df) == 120
assert static_test_summary_df["Test_ID"].nunique() == 120

hysteresis_df = matched_static_hysteresis(static_test_summary_df)
linearity_df, linearity_metrics = linearity_metric(static_test_summary_df)
temperature_influence_df, temperature_metrics = temperature_influence_metric(
    static_test_summary_df
)

print("Matched static hysteresis")
display(
    hysteresis_df.sort_values("Hysteresis_Abs", ascending=False).head(10).round(6)
)

print("Linearity by RH")
display(linearity_df.round(6))
print("Linearity metrics:", linearity_metrics)

print("Temperature influence by RH × Direction")
display(temperature_influence_df.round(6))
print("Temperature metrics:", temperature_metrics)

print("FA-02 static metrics: PASS")

## 5. FA-03 — Integrated Milestone Evidence Audit

In [ ]:
evidence = load_final_evidence(project_root)

for key, table in evidence.items():
    assert len(table) > 0, f"Missing evidence table: {key}"

print("Loaded final evidence tables:")
for key, table in evidence.items():
    print(f"{key:24s}: {len(table):4d} rows")

print("FA-03 integrated milestone evidence audit: PASS")

## 6. FA-04 — Public Requirement Compliance Matrix

Status vocabulary:

- PASS
- FAIL
- NOT_DEMONSTRATED
- INFORMATIONAL

No new threshold is introduced.

In [ ]:
compliance_matrix_df = build_compliance_matrix(
    public_spec=public_spec,
    static_test_summary=static_test_summary_df,
    hysteresis_df=hysteresis_df,
    linearity_metrics=linearity_metrics,
    temp_metrics=temperature_metrics,
    evidence=evidence,
)

display(compliance_matrix_df)

allowed_status = {"PASS", "FAIL", "NOT_DEMONSTRATED", "INFORMATIONAL"}
assert set(compliance_matrix_df["Status"]).issubset(allowed_status)

print("Status counts")
display(
    compliance_matrix_df["Status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Count")
)

print("FA-04 public requirement matrix: PASS")

## 7. FA-05 — Application-Risk Matrix

Some important application risks do not map cleanly onto a single public binary limit.

They are retained separately so the final suitability decision does not hide dynamic limitations.

In [ ]:
application_risk_df = build_application_risk_matrix(evidence)

display(application_risk_df.round(6))

print("FA-05 application-risk matrix: PASS")

## 8. FA-06 — Final Decision Logic

The final decision is intentionally layered.

### Full public-spec compliance

Fully demonstrated only if all mandatory binary requirements are PASS and none are NOT_DEMONSTRATED.

### Static / steady-state HVAC suitability

Evaluates calibrated static performance, uncertainty, repeatability, static hysteresis, linearity, temperature influence, and stability evidence.

### Dynamic HVAC suitability

Evaluates direct response evidence and continuously varying RH behavior.

In [ ]:
binary_rows = compliance_matrix_df.loc[
    compliance_matrix_df["Status"].isin(
        ["PASS", "FAIL", "NOT_DEMONSTRATED"]
    )
].copy()

full_spec_status = (
    "FULLY DEMONSTRATED"
    if (binary_rows["Status"] == "PASS").all()
    else "NOT FULLY DEMONSTRATED"
)

static_required = [
    "Static accuracy",
    "Repeatability sigma",
    "Static hysteresis",
    "Linearity",
    "Temperature influence",
]
static_rows = compliance_matrix_df.loc[
    compliance_matrix_df["Requirement"].isin(static_required)
].copy()

if (static_rows["Status"] == "FAIL").any():
    static_suitability = "CONDITIONALLY SUITABLE"
elif (static_rows["Status"] == "PASS").all():
    static_suitability = "SUITABLE"
else:
    static_suitability = "CONDITIONALLY SUITABLE"

high_dynamic_risk = (
    application_risk_df["Severity"].eq("HIGH").any()
)

dynamic_suitability = (
    "NOT RECOMMENDED AS-IS"
    if high_dynamic_risk
    else "CONDITIONALLY SUITABLE"
)

final_decision_df = pd.DataFrame(
    [
        {
            "Decision_Area": "Full public-spec compliance",
            "Decision": full_spec_status,
            "Basis": (
                "All mandatory public requirements must be directly demonstrated; "
                "NOT_DEMONSTRATED prevents a full-compliance claim."
            ),
        },
        {
            "Decision_Area": "Steady-state/static HVAC suitability",
            "Decision": static_suitability,
            "Basis": (
                "Static accuracy is uncertainty-aware; strict repeatability and "
                "other static requirements are evaluated separately."
            ),
        },
        {
            "Decision_Area": "Dynamic HVAC suitability",
            "Decision": dynamic_suitability,
            "Basis": (
                "Direction-dependent ramp/cyclic behavior, attenuation/phase lag, "
                "and unresolved direct t90 compliance materially limit dynamic use."
            ),
        },
    ]
)

display(final_decision_df)

print("FA-06 final decision logic: PASS")

## 9. FA-07 — Answer to the Project Engineering Question

The formal wording below is generated from the requirement matrix rather than from a single aggregate score.

In [ ]:
failed_requirements = compliance_matrix_df.loc[
    compliance_matrix_df["Status"].eq("FAIL"),
    "Requirement",
].tolist()

not_demonstrated_requirements = compliance_matrix_df.loc[
    compliance_matrix_df["Status"].eq("NOT_DEMONSTRATED"),
    "Requirement",
].tolist()

print("FAIL requirements:")
print(failed_requirements)

print("\nNOT_DEMONSTRATED requirements:")
print(not_demonstrated_requirements)

print("\nFormal final conclusion")
print("-----------------------")
print(
    "The frozen C1-calibrated DUT demonstrates useful steady-state RH measurement "
    "performance under the tested conditions, including uncertainty-aware static "
    "accuracy at the declared evaluation points. However, full public-spec "
    "compliance is not demonstrated. Any failed static requirements, unresolved "
    "annual-drift/t90 compliance, and the strong direction-dependent dynamic "
    "behavior prevent an unconditional suitability claim. The sensor is best "
    "classified as conditionally suitable for slow or steady HVAC monitoring "
    "under the tested conditions, but not recommended as-is for applications "
    "requiring fast, direction-neutral dynamic tracking."
)

print("\nFA-07 final engineering question answered: PASS")

## 10. Curated Figures

In [ ]:
# Compliance status
status_order = ["PASS", "FAIL", "NOT_DEMONSTRATED", "INFORMATIONAL"]
status_counts = (
    compliance_matrix_df["Status"]
    .value_counts()
    .reindex(status_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(status_counts.index, status_counts.values)
ax.set_ylabel("Number of public requirements")
ax.set_title("HV-013 Final Public-Requirement Status")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(
    figures_dir / "hv013_compliance_status.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

# Evidence summary: normalized observed/limit ratios for directly comparable metrics
plot_rows = []

for _, row in compliance_matrix_df.iterrows():
    if row["Requirement"] == "Static accuracy":
        limit = limits["static_accuracy_abs_rh_percent"]
    elif row["Requirement"] == "Repeatability sigma":
        limit = limits["repeatability_sigma_max_rh_percent"]
    elif row["Requirement"] == "Static hysteresis":
        limit = limits["hysteresis_max_rh_percent"]
    elif row["Requirement"] == "Linearity":
        limit = limits["linearity_max_rh_percent"]
    elif row["Requirement"] == "Temperature influence":
        limit = limits["temperature_influence_max_rh_percent_per_c"]
    else:
        continue

    plot_rows.append(
        {
            "Requirement": row["Requirement"],
            "Observed_to_Limit": float(row["Observed"]) / float(limit),
        }
    )

normalized_static_df = pd.DataFrame(plot_rows)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    normalized_static_df["Requirement"],
    normalized_static_df["Observed_to_Limit"],
)
ax.axhline(1.0, linestyle="--", label="Public limit")
ax.set_ylabel("Observed / public limit")
ax.set_title("HV-013 Direct Static Requirement Margin")
ax.tick_params(axis="x", rotation=25)
ax.legend()
fig.tight_layout()
fig.savefig(
    figures_dir / "hv013_final_evidence_summary.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Curated HV-013 figures exported.")

## 11. Curated Evidence Export

In [ ]:
static_metric_details_df = pd.DataFrame(
    [
        {
            "Metric": "Max_TestID_Repeatability_SD",
            "Value": float(evidence["repeatability_detail"]["Std_Error_RH"].max()),
            "Units": "%RH",
        },
        {
            "Metric": "Pooled_Repeatability_SD",
            "Value": float(
                evidence["rr_summary"].loc[
                    evidence["rr_summary"]["Component"].eq(
                        "Within-Test Repeatability"
                    ),
                    "Value_RH",
                ].iloc[0]
            ),
            "Units": "%RH",
        },
        {
            "Metric": "Max_Static_Hysteresis",
            "Value": float(hysteresis_df["Hysteresis_Abs"].max()),
            "Units": "%RH",
        },
        {
            "Metric": "Linearity_Max_Abs_Deviation",
            "Value": float(linearity_metrics["Max_Abs_Deviation"]),
            "Units": "%RH",
        },
        {
            "Metric": "Temperature_Influence_Max_Abs_Slope",
            "Value": float(temperature_metrics["Max_Abs_Slope_RH_per_C"]),
            "Units": "%RH/degC",
        },
    ]
)

compliance_matrix_df.to_csv(
    tables_dir / "hv013_compliance_matrix.csv",
    index=False,
)
static_metric_details_df.to_csv(
    tables_dir / "hv013_static_metric_details.csv",
    index=False,
)
application_risk_df.to_csv(
    tables_dir / "hv013_application_risk_matrix.csv",
    index=False,
)
final_decision_df.to_csv(
    tables_dir / "hv013_final_decision_summary.csv",
    index=False,
)

print("Curated HV-013 tables exported.")

## 12. Initial FA-001 Checkpoint

A clean-kernel Run All should establish:

- FA-01 public acceptance-limit audit PASS;
- FA-02 static metrics PASS;
- FA-03 integrated evidence audit PASS;
- FA-04 public requirement matrix PASS;
- FA-05 application-risk matrix PASS;
- FA-06 final decision logic PASS;
- FA-07 final engineering question answered PASS;
- four curated CSV tables;
- two curated figures.

The next step is final interpretation review and formal FA-001 closure.